# Stage 2a pull: 1-minute CME futures around each CPI release (Databento)

Pulls 1-minute bars from 16:00 ET the business day before each CPI release to 17:00 ET on release day, for:
- ZT (2-year note), ZN (10-year note): volume-rolled front contract
- SR3 (3-month SOFR): 2nd to 5th contracts by expiry, the ones that price the next year of Fed policy
- ES (S&P 500 e-mini), 6E (euro, the dollar's mirror): cross-asset checks

**It checks the cost first and does nothing until you set `CONFIRM = True`.** Expect well under $5 at 1-minute bars; it stops if the quote is above `MAX_SPEND`.
Put your Databento API key in Colab's Secrets panel (key icon on the left) as `DATABENTO_API_KEY`, or paste it when asked.

In [ ]:
MOUNT_DRIVE = True
CONFIRM = False          # set True after you have seen the cost
MAX_SPEND = 10.0         # dollars; refuses to pull above this
SYMBOLS = ["SR3.v.0", "SR3.v.1", "SR3.v.2", "SR3.v.3"]   # SR3 ranked by volume (liquid quarterlies). Earlier run: ZT.v.0, ZN.v.0, SR3.c.1-4, ES.v.0, 6E.v.0 (already saved)
SCHEMA = "ohlcv-1m"
HOLDOUT_MONTHS = 12      # most recent releases flagged as the untouched final test set

!pip -q install databento pyarrow
import pandas as pd, numpy as np, databento as db, time
from pathlib import Path
from getpass import getpass
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RAW = Path("/content/drive/MyDrive/macro-news-pricing/data/raw")
else:
    RAW = Path("/content/data/raw")
OUT = RAW / "futures_1m"; OUT.mkdir(parents=True, exist_ok=True)
try:
    from google.colab import userdata
    KEY = userdata.get("DATABENTO_API_KEY")
except Exception:
    KEY = getpass("Databento API key: ")
client = db.Historical(KEY)

## Event list and windows

In [ ]:
ev = pd.read_parquet(RAW / "qa_events_final.parquet")
ev = ev[~ev.event_ticker.str.contains("CORE")]
ev = ev[~ev.event_ticker.isin({"KXCPI-25OCT", "KXCPI-25NOV"})]
days = sorted(pd.to_datetime(ev.release_date_et).unique())
W = pd.DataFrame({"release_date": days})
W["start"] = [(d - pd.offsets.BDay(1) + pd.Timedelta(hours=16)).tz_localize("America/New_York").tz_convert("UTC") for d in W.release_date]
W["end"] = [(d + pd.Timedelta(hours=17)).tz_localize("America/New_York").tz_convert("UTC") for d in W.release_date]
cut = W.release_date.max() - pd.DateOffset(months=HOLDOUT_MONTHS)
W["holdout"] = W.release_date > cut
W.to_parquet(RAW / "stage2_event_windows.parquet", index=False)
print(f"{len(W)} release days, {W.release_date.min():%Y-%m-%d} to {W.release_date.max():%Y-%m-%d}; "
      f"holdout (do not analyse yet): {int(W.holdout.sum())} days after {cut:%Y-%m-%d}")

## 1. Check symbols and cost (free)

In [ ]:
# symbol check on one window first, so a bad symbol fails cheaply
w0 = W.iloc[-1]
for s in SYMBOLS:
    try:
        c = client.metadata.get_cost(dataset="GLBX.MDP3", symbols=[s], stype_in="continuous", schema=SCHEMA,
                                     start=w0.start.isoformat(), end=w0.end.isoformat())
        print(f"  ok  {s:9s} one window ${c:.4f}")
    except Exception as e:
        print(f"  BAD {s:9s} {str(e)[:150]}")

total = 0.0
for w in W.itertuples():
    total += client.metadata.get_cost(dataset="GLBX.MDP3", symbols=SYMBOLS, stype_in="continuous", schema=SCHEMA,
                                      start=w.start.isoformat(), end=w.end.isoformat())
print(f"\nTotal cost for {len(W)} windows x {len(SYMBOLS)} symbols: ${total:.2f}  (limit ${MAX_SPEND:.2f})")

## 2. Pull (only runs with CONFIRM = True)

In [ ]:
assert CONFIRM, "Check the cost above, then set CONFIRM = True in the first cell and rerun this cell."
assert total <= MAX_SPEND, f"Cost ${total:.2f} is above MAX_SPEND"
for w in W.itertuples():
    for s in SYMBOLS:
        f = OUT / f"{w.release_date:%Y%m%d}_{s.replace('.', '_')}.parquet"
        if f.exists():
            continue                                   # cached: reruns resume and are not charged again
        try:
            d = client.timeseries.get_range(dataset="GLBX.MDP3", symbols=[s], stype_in="continuous", schema=SCHEMA,
                                            start=w.start.isoformat(), end=w.end.isoformat()).to_df()
        except Exception as e:
            print("fail", w.release_date.date(), s, str(e)[:120]); continue
        d = d.reset_index()[["ts_event", "instrument_id", "open", "high", "low", "close", "volume"]]
        d["symbol"], d["release_date"] = s, w.release_date
        d.to_parquet(f, index=False)
    print("done", w.release_date.date())
F = pd.concat([pd.read_parquet(f) for f in sorted(OUT.glob("*.parquet"))], ignore_index=True)
F.to_parquet(RAW / "futures_1m.parquet", index=False)
print(F.groupby("symbol").agg(days=("release_date", "nunique"), bars=("close", "size")).to_string())

## 3. Quick look: 2-year move in the first minutes after 8:30

In [ ]:
F = pd.read_parquet(RAW / "futures_1m.parquet")
F["et"] = pd.to_datetime(F.ts_event, utc=True).dt.tz_convert("America/New_York")
def px(g, hh, mm):   # last close at or before hh:mm ET on release day
    t = g.release_date.iloc[0].tz_localize("America/New_York") + pd.Timedelta(hours=hh, minutes=mm)
    x = g[g.et <= t]
    return x.close.iloc[-1] if len(x) else np.nan
zt = F[F.symbol == "ZT.v.0"].groupby("release_date")
R = pd.DataFrame({"p0829": zt.apply(lambda g: px(g, 8, 29)), "p0835": zt.apply(lambda g: px(g, 8, 35)),
                  "p0930": zt.apply(lambda g: px(g, 9, 30)), "p1600": zt.apply(lambda g: px(g, 16, 0))})
R["jump"], R["drift_to_0930"], R["drift_to_1600"] = R.p0835 - R.p0829, R.p0930 - R.p0835, R.p1600 - R.p0930
print("ZT price changes (points; 1 point ~ 50bp of 2-year yield):")
print(R[["jump", "drift_to_0930", "drift_to_1600"]].describe().round(4).to_string())
print("corr(jump, later drift):", R[["jump", "drift_to_0930", "drift_to_1600"]].corr().round(3).iloc[0, 1:].to_dict())